# Exploration Livres : Open Library + Google Books

Contrairement à Open Products Facts (dump massif), on utilise ici directement les **API** :
- **Open Library** : recherche + détail par œuvre (le détail contient souvent une vraie description)
- **Google Books** : comparatif, souvent plus complet sur les résumés

Plan :
1. Recherche d'un échantillon de livres (mot-clé / sujet configurable)
2. Récupération du détail de chaque oeuvre (description incluse) via Open Library
3. Calcul du taux de remplissage réel (titre / description / ISBN)
4. Comparatif avec Google Books sur le même échantillon
5. Export final : designation / description / code_produit

In [ ]:
# !pip install requests pandas tqdm

## 1. Configuration

In [ ]:
import requests
import time
import pandas as pd
from tqdm import tqdm

REQUETE = "fiction"      # mot-clé ou sujet à rechercher (ex: "science fiction", "cuisine", "histoire de france")
TAILLE_ECHANTILLON = 500  # nombre de livres à récupérer pour l'analyse
PAGE_SIZE = 100           # résultats par page (max 100 côté Open Library)

HEADERS = {"User-Agent": "MonScriptExploration/1.0 (contact@example.com)"}

## 2. Recherche d'un échantillon via l'API Search d'Open Library

Endpoint : `https://openlibrary.org/search.json?q=...`

On récupère : la clé de l'oeuvre (`key`, ex: `/works/OL45804W`), le titre, et l'ISBN si présent.

In [ ]:
def rechercher_livres(requete: str, taille: int, page_size: int = 100):
    resultats = []
    page = 1
    with tqdm(total=taille, desc="Recherche") as bar:
        while len(resultats) < taille:
            params = {
                "q": requete,
                "page": page,
                "limit": page_size,
                "fields": "key,title,isbn,first_publish_year,author_name,subject",
            }
            r = requests.get("https://openlibrary.org/search.json", params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            docs = r.json().get("docs", [])
            if not docs:
                break
            resultats.extend(docs)
            bar.update(len(docs))
            page += 1
            time.sleep(0.2)  # on reste poli avec l'API
    return resultats[:taille]

docs = rechercher_livres(REQUETE, TAILLE_ECHANTILLON, PAGE_SIZE)
print(f"{len(docs)} livres trouvés pour la requête '{REQUETE}'.")
docs[0] if docs else None

## 3. Récupération du détail de chaque oeuvre (avec description)

Endpoint : `https://openlibrary.org{key}.json` — contient un champ `description` (texte ou objet `{"type":..., "value":...}` selon les cas).

In [ ]:
def extraire_description(data: dict) -> str:
    desc = data.get("description")
    if desc is None:
        return ""
    if isinstance(desc, dict):
        return desc.get("value", "")
    return str(desc)


def recuperer_details(docs: list):
    lignes = []
    for doc in tqdm(docs, desc="Détail des oeuvres"):
        key = doc.get("key")
        if not key:
            continue
        try:
            r = requests.get(f"https://openlibrary.org{key}.json", headers=HEADERS, timeout=30)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            data = {}
        isbn_list = doc.get("isbn", [])
        lignes.append({
            "key": key,
            "title": doc.get("title", ""),
            "description": extraire_description(data),
            "isbn": isbn_list[0] if isbn_list else "",
            "author_name": ", ".join(doc.get("author_name", []) or []),
            "first_publish_year": doc.get("first_publish_year", ""),
            "subject": ", ".join((doc.get("subject") or [])[:5]),
        })
        time.sleep(0.1)  # limite de charge sur l'API, ajustez si besoin
    return pd.DataFrame(lignes)

df = recuperer_details(docs)
df.head(10)

## 4. Taux de remplissage réel

In [ ]:
total = len(df)
print(f"Total analysé : {total}\n")

for col in ["title", "description", "isbn", "author_name"]:
    rempli = (df[col].fillna("").str.strip() != "").sum()
    print(f"{col:20s} : {rempli}/{total} remplis ({rempli/total*100:.1f}%)")

In [ ]:
# Aperçu de vraies descriptions (pour juger de la qualité, pas juste la présence)
avec_description = df[df["description"].fillna("").str.strip() != ""]
print(f"{len(avec_description)} livres avec une description non vide.\n")
for _, row in avec_description.head(5).iterrows():
    print("---")
    print("Titre :", row["title"])
    print("Description :", row["description"][:300], "..." if len(row["description"]) > 300 else "")

## 5. Comparatif avec Google Books (sur le même échantillon)

Endpoint public, sans clé nécessaire pour un usage léger : `https://www.googleapis.com/books/v1/volumes?q=...`

⚠️ Sans clé API, le quota est limité (bien pour tester ; pour un usage massif il faut une clé Google Cloud).

In [ ]:
def chercher_google_books(titre: str, auteur: str = ""):
    q = f"intitle:{titre}"
    if auteur:
        q += f"+inauthor:{auteur.split(',')[0]}"
    try:
        r = requests.get(
            "https://www.googleapis.com/books/v1/volumes",
            params={"q": q, "maxResults": 1},
            timeout=30,
        )
        r.raise_for_status()
        items = r.json().get("items", [])
        if not items:
            return ""
        return items[0].get("volumeInfo", {}).get("description", "")
    except Exception:
        return ""

# Comparatif sur un sous-échantillon (pour ne pas trop taper l'API sans clé)
SOUS_ECHANTILLON = min(50, len(df))
comparatif = df.head(SOUS_ECHANTILLON).copy()

tqdm.pandas(desc="Google Books")
comparatif["description_google"] = comparatif.progress_apply(
    lambda row: chercher_google_books(row["title"], row["author_name"]), axis=1
)

rempli_ol = (comparatif["description"].fillna("").str.strip() != "").sum()
rempli_gb = (comparatif["description_google"].fillna("").str.strip() != "").sum()
print(f"Sur {SOUS_ECHANTILLON} livres :")
print(f"  Open Library  : {rempli_ol} descriptions non vides ({rempli_ol/SOUS_ECHANTILLON*100:.1f}%)")
print(f"  Google Books  : {rempli_gb} descriptions non vides ({rempli_gb/SOUS_ECHANTILLON*100:.1f}%)")

## 6. Export final (designation / description / code_produit)

On privilégie la description Open Library, et on complète avec Google Books si elle est vide (sur le sous-échantillon comparé ; à étendre à tout l'échantillon si Google Books s'avère nettement meilleur).

In [ ]:
final = pd.DataFrame({
    "designation": df["title"],
    "description": df["description"],
    "code_produit": df["isbn"],
})

final = final[final["designation"].str.strip() != ""]
final = final.drop_duplicates(subset="designation")

final.to_csv("livres_extraits.csv", index=False, encoding="utf-8")
print(f"{len(final)} livres exportés dans livres_extraits.csv")
final.head(10)